# typology2 student -> one Q4_K_M GGUF for Ollama, then a dev-only quantisation check

Run top to bottom on a Colab **L4** (Runtime > Change runtime type).
1. **Merge:** load Qwen2.5-7B-Instruct in bf16 and merge the frozen `typology2` LoRA adapter into it.
2. **Convert:** turn the merged model into a GGUF with llama.cpp, then quantise it to **Q4_K_M**.
3. **Save:** copy **only the Q4_K_M GGUF** (about 4.7 GB) to Drive. The merged model and the f16 GGUF stay on Colab's local disk and are deleted as soon as each step is done.
4. **Dev check:** serve the GGUF with Ollama on the L4, using the same `Modelfile` as on the Mac, and write notes for the **50 dev alerts only**. Test is never touched.

**Estimate (not measured):** 45-75 min, about 1-1.25 L4 hours (~2-2.5 compute units).
- Download 15 GB: 3-5 min
- Merge and save: 5 min
- Build llama.cpp's quantiser: 3-5 min
- Convert to f16 GGUF: 5-8 min
- Quantise: 5-10 min
- Copy to Drive: 2-4 min
- Dev notes: 50 x 15-20 s = 15-20 min

**Disk:**
- **Colab local:** peaks at about 31 GB (the base download plus the merged model, or the merged model plus the f16 GGUF).
- **Drive:** needs about 5 GB free.

**Drive folder `aml_finetune/` must hold:**
- `adapter_Qwen2.5-7B-Instruct_typology2/` (already there)
- `dev_inputs.jsonl` (already there)

**Upload from the repo:**
- `src/ollama_generate.py`
- `ollama/Modelfile`

In [ ]:
# ---- Settings (change here, nowhere else) ----
import os, shutil, subprocess, time, json, urllib.request

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"          # the model the adapter was trained on
DRIVE_DIR = "/content/drive/MyDrive/aml_finetune"
ADAPTER_DIR = DRIVE_DIR + "/adapter_Qwen2.5-7B-Instruct_typology2"   # frozen, read only
GGUF_NAME = "aml-student-typology2-Q4_K_M.gguf"
GGUF_ON_DRIVE = DRIVE_DIR + "/" + GGUF_NAME
DEV_RAW = DRIVE_DIR + "/dev_raw_Qwen2.5-7B-Instruct_typology2_q4km.jsonl"  # dev only
OLLAMA_MODEL = "aml-student-typology2"
WORK = "/content/gguf"                  # local disk: merged and f16 files never go to Drive
MERGED_DIR = WORK + "/merged"
F16_GGUF = WORK + "/student-f16.gguf"
Q4_GGUF = WORK + "/" + GGUF_NAME
os.makedirs(WORK, exist_ok=True)


def run(command):
    """Run a shell command, show its output as it arrives, and stop if it fails."""
    process = subprocess.Popen(["bash", "-o", "pipefail", "-c", command],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed: {command}")


def free_gb(path="/content"):
    return shutil.disk_usage(path).free / 1e9

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
run("nvidia-smi --query-gpu=name,memory.total --format=csv")
for name in ["adapter_Qwen2.5-7B-Instruct_typology2/adapter_config.json", "dev_inputs.jsonl",
             "ollama_generate.py", "Modelfile"]:
    assert os.path.exists(os.path.join(DRIVE_DIR, name)), "missing in Drive: " + name
print("Local disk free: %.0f GB (need about 31)" % free_gb())
print("Drive free: %.0f GB (need about 5)" % free_gb("/content/drive/MyDrive"))
SKIP_BUILD = os.path.exists(GGUF_ON_DRIVE)
print("GGUF already on Drive, skipping merge/convert/quantise" if SKIP_BUILD else "Will build the GGUF")

In [ ]:
# Same library versions as the training notebook (finetune_colab.ipynb).
!pip -q install transformers==5.24.1 peft==0.22.0 accelerate==1.16.0 sentencepiece

**Merge.** The adapter was trained on a 4-bit (NF4) base (QLoRA). Here it is merged into the bf16 base, the usual way to export, so the merged weights differ slightly from what training saw. Quantising to Q4_K_M changes them a little more. The dev check below measures the combined effect.

In [ ]:
if not SKIP_BUILD:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    start = time.time()
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map="cuda")
    merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")
    AutoTokenizer.from_pretrained(ADAPTER_DIR).save_pretrained(MERGED_DIR)  # as trained
    del base, merged
    torch.cuda.empty_cache()
    # The base download is no longer needed: free its ~15 GB before converting.
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct"),
                  ignore_errors=True)
    print(f"Merged in {(time.time() - start) / 60:.1f} min; local disk free {free_gb():.0f} GB")

In [ ]:
# llama.cpp: the converter (Python) and only the quantiser binary (CPU build is enough).
if not SKIP_BUILD:
    run(f"rm -rf {WORK}/llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp {WORK}/llama.cpp")
    run(f"git -C {WORK}/llama.cpp log -1 --format='llama.cpp %h %cd'")
    run(f"pip -q install {WORK}/llama.cpp/gguf-py")
    run(f"cd {WORK}/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null "
        f"&& cmake --build build --target llama-quantize -j > /dev/null")

In [ ]:
if not SKIP_BUILD:
    start = time.time()
    run(f"python {WORK}/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outtype f16 --outfile {F16_GGUF}")
    shutil.rmtree(MERGED_DIR)                       # merged fp16 model: never kept
    run(f"{WORK}/llama.cpp/build/bin/llama-quantize {F16_GGUF} {Q4_GGUF} Q4_K_M")
    os.remove(F16_GGUF)                             # f16 GGUF: never kept
    shutil.copy(Q4_GGUF, GGUF_ON_DRIVE)             # the only model file saved to Drive
    digest = subprocess.check_output(["sha256sum", Q4_GGUF], text=True).split()[0]
    open(GGUF_ON_DRIVE + ".sha256", "w").write(digest + "\n")  # to check the Mac download
    print("sha256", digest)
    print(f"Q4_K_M: {os.path.getsize(Q4_GGUF) / 1e9:.2f} GB -> {GGUF_ON_DRIVE} "
          f"({(time.time() - start) / 60:.1f} min)")
else:
    shutil.copy(GGUF_ON_DRIVE, Q4_GGUF)

**Dev check with Ollama on the L4.** This uses the same `Modelfile` as the Mac, with only its `FROM` path changed to this local GGUF. Notes are written for the 50 dev alerts only.

In [ ]:
run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh")
env = dict(os.environ, OLLAMA_MODELS="/content/ollama_models")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
lines = open(DRIVE_DIR + "/Modelfile").read().splitlines()
lines = [("FROM " + Q4_GGUF) if line.startswith("FROM ") else line for line in lines]
open(WORK + "/Modelfile", "w").write("\n".join(lines) + "\n")
run(f"OLLAMA_MODELS=/content/ollama_models ollama create {OLLAMA_MODEL} -f {WORK}/Modelfile")
run("ollama list")

In [ ]:
assert "dev_" in os.path.basename(DEV_RAW)   # dev only: test stays the adapter's one scored run
start = time.time()
run(f"python {DRIVE_DIR}/ollama_generate.py --inputs {DRIVE_DIR}/dev_inputs.jsonl --output {DEV_RAW} --model {OLLAMA_MODEL}")
print(f"Dev notes took {(time.time() - start) / 60:.1f} min")

**Then on the Mac:**
1. Download `dev_raw_Qwen2.5-7B-Instruct_typology2_q4km.jsonl` to `data/finetune/`.
2. Run `python -m src.quant_check --gguf-raw data/finetune/dev_raw_Qwen2.5-7B-Instruct_typology2_q4km.jsonl`.

It compares the GGUF with the adapter on dev: JSON validity, invented facts and typology agreement. The pass rules are fixed in `config.QUANT_*`. Then follow `docs/OFFLINE_DEMO.md`.